# Calenda — Qwen3-0.6B LoRA 학습 (Kaggle · T4 x1 / T4 x2)

단일 베이스 **Qwen/Qwen3-0.6B** · **completion-only 손실 ON**.
T4 x2 선택 시 torchrun DDP 자동 적용 (셀 6에서 GPU 수 감지).

**사전 설정 (노트북 실행 전)**
- Notebook settings → Accelerator: **GPU T4 x1** 또는 **T4 x2**
- Add-ons → Secrets: `HF_TOKEN` 등록 (HuggingFace write token)
- Settings → Internet: **On**

위→아래 순서 실행.

## 0. 환경 초기화

In [ ]:
import os

# Kaggle Secrets에서 HF_TOKEN 로드
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    _hf_token = ''

os.environ['HF_TOKEN'] = _hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = _hf_token
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
print('HF_TOKEN:', '있음' if _hf_token else '없음 (LoRA HF 업로드 스킵됨)')

## 0-1. HF_TOKEN 검증

In [ ]:
from huggingface_hub import HfApi
import os

_token = os.environ.get('HF_TOKEN', '')
if not _token:
    print('HF_TOKEN 없음 — Secrets에 HF_TOKEN이 등록됐는지 확인하세요.')
else:
    try:
        _info = HfApi(token=_token).whoami()
        print(f'HF_TOKEN 정상 — 사용자: {_info["name"]} / 이메일: {_info.get("email", "비공개")}')
    except Exception as e:
        print(f'HF_TOKEN 인증 실패: {e}')

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. repo clone + 버전 배너

In [ ]:
import os, yaml, json, subprocess
from collections import Counter

WORK = '/kaggle/working'
REPO = f'{WORK}/Calenda'

%cd {WORK}
if not os.path.exists(REPO):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd {REPO} && git fetch origin && git reset --hard origin/main
%cd {REPO}

_c = yaml.safe_load(open('configs/train_qwen3_0_6b.yaml', encoding='utf-8'))
def _stat(p):
    rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
    c = Counter(str(r['gold'].get('schedule_status')) for r in rows)
    return len(rows), c.get('yes', 0), c.get('no', 0)
_tt, _ty, _tn = _stat('data/processed/train.jsonl')
_gt, _gy, _gn = _stat('data/eval/golden.jsonl')
_head = subprocess.run('git log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip()
print('=' * 62)
print(f"  라운드          : {_c['run_name']}  ·  epochs={_c['num_train_epochs']}")
print(f"  completion-only : {_c.get('completion_only_loss', False)}")
print(f"  컴밋            : {_head[:60]}")
print(f"  학습셋          : train {_tt}   yes {_ty} / no {_tn}")
print(f"  평가셋          : golden {_gt}   yes {_gy} / no {_gn}")
print('=' * 62)

## 3. 의존성 설치

`cuda: True` 확인 후 진행.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y -q 2>/dev/null || true
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'ngpu:', torch.cuda.device_count())

## 4. 데이터 확인

In [ ]:
import os
for p in ['data/processed/train.jsonl', 'data/eval/golden.jsonl']:
    n = sum(1 for l in open(p, encoding='utf-8') if l.strip())
    print(f'{p}: {n} records')
_val = 'data/processed/val.jsonl'
if os.path.isfile(_val):
    print(f'{_val}: {sum(1 for l in open(_val, encoding="utf-8") if l.strip())} records')
else:
    print(f'{_val}: 없음')

## 5. 사전점검 — 학습 렌더 == 추론 프롬프트 + gold JSON

`OK` 확인 후 학습 진행.

In [ ]:
import os, yaml, json
from transformers import AutoTokenizer

CONFIG = 'configs/train_qwen3_0_6b.yaml'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"라운드 {_cfg['run_name']} | base {_mcfg['hf_id']}")
print(f"output_dir {_cfg['output_dir']} | 유효배치 {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']}")

_tok = AutoTokenizer.from_pretrained(
    _mcfg['hf_id'],
    trust_remote_code=_mcfg.get('trust_remote_code', False),
    token=False,
)
_sys  = _mcfg['system_prompt']
_user = '<채널: KakaoTalk>\n<메시지>\n내일 3시 회의\n</메시지>'
_gold = json.dumps({'schedule_status': 'no', 'events': []}, ensure_ascii=False)
_full = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _user}, {'role': 'assistant', 'content': _gold}]
_pre  = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _user}]
_train_str = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
_extra = {'enable_thinking': False} if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or '') else {}
_infer_str = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
_aligned   = _train_str.startswith(_infer_str)
_json_next = (_train_str[len(_infer_str):] if _aligned else '').lstrip().startswith('{')
_rt_ok     = '<|im_start|>assistant\n' in _train_str
print(f'정렬: {_aligned} | JSON바로시작: {_json_next} | response_template: {_rt_ok}',
      '→ OK' if (_aligned and _json_next and _rt_ok) else '→ FAIL')
assert _aligned and _json_next, '학습/추론 프롬프트 정렬 깨짐'

## 6. 학습

In [ ]:
import torch
_ngpu = torch.cuda.device_count()
if _ngpu >= 2:
    print(f"학습 시작 → {_cfg['run_name']} · T4 x{_ngpu} (torchrun DDP)")
    !torchrun --nproc_per_node={_ngpu} scripts/train_lora.py --config {CONFIG}
else:
    print(f"학습 시작 → {_cfg['run_name']} · 단일 GPU")
    !python scripts/train_lora.py --config {CONFIG}

## 7. 평가 (LoRA merge → 골든셋 평가)

In [ ]:
import yaml, os
CONFIG = 'configs/train_qwen3_0_6b.yaml'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))

BASE       = _mcfg['hf_id']
LORA_DIR   = _cfg['output_dir']
NAME       = os.path.basename(LORA_DIR)
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON  = f'logs/eval_{NAME}.json'
GOLDEN     = _cfg.get('eval_golden', 'data/eval/golden.jsonl')

print(f'lora: {LORA_DIR} → merged: {MERGED_DIR}')
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON} --model_config {_cfg['model_config']}

print(f'\n--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 8. HuggingFace 백업 (어댑터 + eval + failures)

학습 결과를 `sooryong9885/Calenda-Qwen3-0.6B` private repo에 저장합니다.
HF_TOKEN Secret이 등록된 경우 자동 실행됩니다.

In [ ]:
import os, shutil
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get('HF_TOKEN', '')
REPO_ID  = 'sooryong9885/Calenda-Qwen3-0.6B'

if not HF_TOKEN:
    print('HF_TOKEN 없음 — Secrets에 HF_TOKEN이 등록됐는지 확인하세요.')
else:
    api = HfApi(token=HF_TOKEN)

    # 8-1. LoRA 어댑터 업로드
    print(f'[1/3] LoRA 어댑터 업로드 중 → {REPO_ID}/{NAME}')
    api.upload_folder(
        folder_path=LORA_DIR,
        repo_id=REPO_ID,
        repo_type='model',
        path_in_repo=NAME,
        commit_message=f'upload adapter {NAME}',
    )
    print('      완료')

    # 8-2. eval JSON 업로드
    print(f'[2/3] eval JSON 업로드 중 → {NAME}/eval.json')
    api.upload_file(
        path_or_fileobj=EVAL_JSON,
        path_in_repo=f'{NAME}/eval.json',
        repo_id=REPO_ID,
        repo_type='model',
        commit_message=f'upload eval {NAME}',
    )
    print('      완료')

    # 8-3. failures 업로드
    FAILURES_SRC = 'data/failures/round_latest.jsonl'
    if os.path.isfile(FAILURES_SRC):
        n_fail = sum(1 for l in open(FAILURES_SRC) if l.strip())
        print(f'[3/3] failures 업로드 중 ({n_fail}건) → {NAME}/failures.jsonl')
        api.upload_file(
            path_or_fileobj=FAILURES_SRC,
            path_in_repo=f'{NAME}/failures.jsonl',
            repo_id=REPO_ID,
            repo_type='model',
            commit_message=f'upload failures {NAME}',
        )
        print('      완료')
    else:
        print('[3/3] failures 없음 (전 샘플 통과) — 스킵')

    print(f'\nHF 백업 완료: https://huggingface.co/{REPO_ID}/tree/main/{NAME}')